### Import

In [13]:
import pinocchio as pin  ##type: ignore
from pinocchio.visualize import MeshcatVisualizer  ##type: ignore
import numpy as np
import sys
from scipy.sparse import save_npz, load_npz
from scipy.sparse.csgraph import connected_components

from self_collision import build_link_capsules, default_excluded_pairs
from external_collision import Sphere, check_external_collision, filter_external_collision_free_parallel
from prm_roadmap import build_prm_graph_with_obstacles_parallel
from prm_sampling import sample_batch
from workspace import is_pose_reachable
from prm_query import embed_X_in_graph, dijkstra_path
from simulators import simulate_lqr_from_path

### Load model

In [14]:
model, collision_model, visual_model = pin.buildModelsFromUrdf(
    "lbr_iiwa7_r800.urdf", package_dirs="."
)
data = model.createData()
capsules = build_link_capsules(collision_model)

### Initialize visualizer (meshcat)

In [15]:
try:
    viz = MeshcatVisualizer(model, collision_model, visual_model)
    viz.initViewer(open=True)
    viz.loadViewerModel()
except ImportError as err:
    print(
        "Error while initializing the viewer. "
        "It seems you should install Python meshcat"
    )
    print(err)
    sys.exit(0)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7008/static/


### Load saved internal-collision-free q pool

In [ ]:
loaded = np.load("data/q_pool_internal_collision_free.npz")
q_pool_internal_collision_free = loaded["q_pool"]
print(f"loaded {q_pool_internal_collision_free.shape[0]} internal-collision-free q pool nodes from data/q_pool_internal_collision_free.npz")

### Define sphere obstacles

In [16]:
# obstacles -- verified against a real 200k-sample: no single sphere blocks
# more than ~10% alone, combined survival ~45% of the internal-collision-free pool
spheres = [
    Sphere([0.5, 0.0, 0.5], 0.08),
    Sphere([-0.5, 0.3, 0.5], 0.08),
    Sphere([0.4, 0.4, 0.4], 0.08),
    Sphere([-0.4, -0.4, 0.6], 0.08),
    Sphere([0.0, 0.5, 0.3], 0.08),
    Sphere([0.3, -0.5, 0.5], 0.08),
    Sphere([-0.3, 0.2, 0.8], 0.08),
    Sphere([0.6, -0.2, 0.6], 0.07),
    Sphere([-0.2, -0.5, 0.4], 0.08),
    Sphere([0.2, 0.3, 0.7], 0.07),
]

### Render sphere obstacles in meshcat

In [17]:
import meshcat.geometry as mg
import meshcat.transformations as mtf

for i, sphere in enumerate(spheres):
    viz.viewer[f"obstacles/sphere_{i}"].set_object(
        mg.Sphere(sphere.radius),
        mg.MeshLambertMaterial(color=0xff0000, transparent=True, opacity=0.6),
    )
    viz.viewer[f"obstacles/sphere_{i}"].set_transform(mtf.translation_matrix(sphere.position))

### Helper: draw a green arrow at a task-space waypoint (position + R)

In [18]:
# meshcat's Cylinder is aligned along its local +Y axis by default -- this
# remap rotates that onto local +Z, which is the "approach axis" convention
# our IK uses (R[:, 2]), so the arrow points the same way analytical_ik
# treats the pose's orientation.
_arrow_remap = mtf.rotation_matrix(np.pi / 2, [1, 0, 0])[:3, :3]

def draw_pose_arrow(viz, name, position, R, length=0.12, shaft_radius=0.006, head_radius=0.017, head_length=0.035, color=0x00ff00):
    """Draw a green arrow at (position, R): a thin shaft plus a fatter head
    near the tip (meshcat has no native cone), both along R's approach axis
    (R[:, 2]) -- lets you see each task-space waypoint's position and facing
    direction in the scene, and watch the arm reach for it during playback.
    """
    material = mg.MeshLambertMaterial(color=color)
    shaft_len = length - head_length

    viz.viewer[name]["shaft"].set_object(mg.Cylinder(shaft_len, shaft_radius), material)
    shaft_T = np.eye(4)
    shaft_T[:3, :3] = _arrow_remap
    shaft_T[:3, 3] = _arrow_remap @ np.array([0, shaft_len / 2, 0])
    viz.viewer[name]["shaft"].set_transform(shaft_T)

    viz.viewer[name]["head"].set_object(mg.Cylinder(head_length, head_radius), material)
    head_T = np.eye(4)
    head_T[:3, :3] = _arrow_remap
    head_T[:3, 3] = _arrow_remap @ np.array([0, shaft_len + head_length / 2, 0])
    viz.viewer[name]["head"].set_transform(head_T)

    group_T = np.eye(4)
    group_T[:3, :3] = R
    group_T[:3, 3] = position
    viz.viewer[name].set_transform(group_T)

### Filter q pool for external (sphere-obstacle) collision-free configs

In [ ]:
q_pool_external_collision_free, n_external_collision_free = filter_external_collision_free_parallel(
    q_pool_internal_collision_free, "lbr_iiwa7_r800.urdf", spheres
)
print(f"{n_external_collision_free}/{q_pool_internal_collision_free.shape[0]} q's are external-collision-free (already internal-collision-free)")

### Save external-collision-free q pool to data/

In [ ]:
np.savez_compressed("data/q_pool_external_collision_free.npz", q_pool=q_pool_external_collision_free)
print(f"saved {q_pool_external_collision_free.shape[0]} external-collision-free q's to data/q_pool_external_collision_free.npz")

### Load saved external-collision-free q pool

In [ ]:
loaded = np.load("data/q_pool_external_collision_free.npz")
q_pool_external_collision_free = loaded["q_pool"]
print(f"loaded {q_pool_external_collision_free.shape[0]} external-collision-free q pool nodes from data/q_pool_external_collision_free.npz")

### Build PRM graph (k-NN candidate edges, internal + external collision-free, parallel)

In [ ]:
from scipy.sparse.csgraph import connected_components

k = 10
prm_graph_obstacles = build_prm_graph_with_obstacles_parallel(
    q_pool_external_collision_free, "lbr_iiwa7_r800.urdf", spheres, k=k
)
print(f"built PRM graph: {prm_graph_obstacles.shape[0]} nodes, {prm_graph_obstacles.nnz // 2} edges (k={k})")

n_components, labels = connected_components(prm_graph_obstacles, directed=False)
degrees = np.diff(prm_graph_obstacles.indptr)
largest = np.bincount(labels).max()
print(f"connected components: {n_components}, largest: {largest}/{prm_graph_obstacles.shape[0]} "
      f"({largest / prm_graph_obstacles.shape[0]:.2%}), isolated nodes: {(degrees == 0).sum()}")

### Save PRM graph (with obstacles) to data/

In [ ]:
save_npz("data/prm_graph_obstacles.npz", prm_graph_obstacles)
print(f"saved PRM graph ({prm_graph_obstacles.shape[0]} nodes, {prm_graph_obstacles.nnz // 2} edges) to data/prm_graph_obstacles.npz")

### Load saved PRM graph (with obstacles)

In [19]:
prm_graph_obstacles = load_npz("data/prm_graph_obstacles.npz")
print(f"loaded PRM graph: {prm_graph_obstacles.shape[0]} nodes, {prm_graph_obstacles.nnz // 2} edges from data/prm_graph_obstacles.npz")

loaded PRM graph: 1670438 nodes, 10026605 edges from data/prm_graph_obstacles.npz


### Random start/goal X -> embed -> Dijkstra -> spline -> TVLQR -> simulate (obstacle-aware, fails safely if disconnected)

In [ ]:
rng = np.random.default_rng()
excluded_pairs = default_excluded_pairs(len(capsules))

def random_valid_X():
    while True:
        xyz, R, n_acc = sample_batch(1, model, rng=rng)
        if n_acc and is_pose_reachable(model, data, xyz[0], R[0]):
            return xyz[0], R[0]

# not all q's are connected in this graph (172 isolated nodes, 187 components) --
# a randomly embedded start/goal pair can land in different components, so
# dijkstra_path returns (None, inf) rather than a path in that case. Retry with
# a fresh pair instead of crashing, bounded so this can't loop forever.
max_attempts = 20
path = None
for attempt in range(max_attempts):
    X_pair = [random_valid_X(), random_valid_X()]
    aug_graph, aug_q_pool, node_idxs = embed_X_in_graph(
        X_pair, prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs, spheres=spheres
    )
    while not node_idxs[0] or not node_idxs[1]:
        if not node_idxs[0]:
            X_pair[0] = random_valid_X()
        if not node_idxs[1]:
            X_pair[1] = random_valid_X()
        aug_graph, aug_q_pool, node_idxs = embed_X_in_graph(
            X_pair, prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs, spheres=spheres
        )

    start_idx, goal_idx = node_idxs[0][0], node_idxs[1][0]
    path, dist = dijkstra_path(aug_graph, start_idx, goal_idx)
    if path is not None:
        break
    print(f"attempt {attempt + 1}: start/goal landed in disconnected components, retrying with new X's")

if path is None:
    raise RuntimeError(f"no path found after {max_attempts} attempts -- start/goal kept landing in disconnected graph components")

print(f"start X: {X_pair[0][0]}, goal X: {X_pair[1][0]}")
print(f"dijkstra path: {len(path)} nodes, distance {dist:.4f}")

use_shortcutting = False  # shortcutting can make the cubic spline overshoot through self-collision on sparse waypoints
q_path = aug_q_pool[path]
result = simulate_lqr_from_path(
    model, q_path, T=len(path) * 0.5, dt=0.01, viz=viz, viz_every=5, real_time=True,
    use_shortcutting=use_shortcutting, capsules=capsules, excluded_pairs=excluded_pairs, spheres=spheres,
)
print(f"simulated {len(result['t'])} steps, max tracking error {result['errors'].max():.4f}, final error {result['errors'][-1]:.4f}")

### Chain multiple X waypoints (A->B->C->D->E) into one long obstacle-avoiding sim

In [ ]:
# 5 waypoints with big low/high, left/right swings -- front-right down low,
# back-left up high, back-right down low, front-left up high, then a
# dramatic overhead finish near max reach. Threads through the obstacle
# field on every leg instead of just touring flat corners.
# Each target is snapped to the nearest already-IK-verified real X
# (data/valid_X.npz), falling back to the next-nearest candidate if the
# closest one doesn't survive the internal+external collision filter when
# embedded (guarantees every waypoint is actually usable, not just close).
excluded_pairs = default_excluded_pairs(len(capsules))
loaded_X = np.load("data/valid_X.npz")
positions, rotations = loaded_X["positions"], loaded_X["rotations"]

targets = np.array([
    [0.55, 0.45, 0.15],
    [-0.45, 0.4, 0.85],
    [0.5, -0.55, 0.2],
    [-0.5, -0.4, 0.85],
    [0.05, 0.05, 1.05],
])

X_waypoints = []
for t in targets:
    order = np.argsort(np.linalg.norm(positions - t, axis=1))
    for cand_idx in order[:30]:
        X_cand = (positions[cand_idx], rotations[cand_idx])
        _, _, node_idxs = embed_X_in_graph(
            [X_cand], prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs, spheres=spheres
        )
        if node_idxs[0]:
            X_waypoints.append(X_cand)
            break
    else:
        raise RuntimeError(f"no embeddable X found near target {t} in the 30 nearest candidates")

# embed all 5 together (shared graph copy across every segment below)
aug_graph, aug_q_pool, node_idxs_per_X = embed_X_in_graph(
    X_waypoints, prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs, spheres=spheres
)
print("embedded nodes per waypoint:", [len(idxs) for idxs in node_idxs_per_X])

# chain consecutive waypoints via Dijkstra. Each segment must continue from
# EXACTLY the node the previous segment arrived at -- picking the arrival and
# departure candidates independently can leave two unrelated (and sometimes
# disconnected, or exactly-duplicate) nodes adjacent in the path with no real
# edge between them. Only the very first segment gets to search both sides.
full_path = None
current_node = None
for a in node_idxs_per_X[0]:
    for b in node_idxs_per_X[1]:
        path, dist = dijkstra_path(aug_graph, a, b)
        if path is not None:
            full_path = list(path)
            current_node = b
            print(f"segment 0->1: {len(path)} nodes, distance {dist:.4f}")
            break
    if full_path is not None:
        break
if full_path is None:
    raise RuntimeError("segment 0->1 disconnected across every node candidate tried")

for i in range(1, len(X_waypoints) - 1):
    connected = False
    for b in node_idxs_per_X[i + 1]:
        path, dist = dijkstra_path(aug_graph, current_node, b)
        if path is not None:
            full_path.extend(path[1:])
            current_node = b
            print(f"segment {i}->{i + 1}: {len(path)} nodes, distance {dist:.4f}")
            connected = True
            break
    if not connected:
        raise RuntimeError(f"segment {i}->{i + 1} disconnected from arrival node {current_node} across every candidate tried")

print(f"total chained path: {len(full_path)} nodes")

use_shortcutting = False  # shortcutting can make the cubic spline overshoot through self-collision on sparse waypoints
q_path = aug_q_pool[full_path]
result = simulate_lqr_from_path(
    model, q_path, T=len(full_path) * 0.5, dt=0.01, viz=viz, viz_every=5, real_time=True,
    use_shortcutting=use_shortcutting, capsules=capsules, excluded_pairs=excluded_pairs, spheres=spheres,
)
print(f"simulated {len(result['t'])} steps, max tracking error {result['errors'].max():.4f}, final error {result['errors'][-1]:.4f}")

### Draw green arrows at the 5 tour waypoints

In [ ]:
for i, (p, R) in enumerate(X_waypoints):
    draw_pose_arrow(viz, f"waypoints/{i}", p, R)

### Helix trajectory that avoids all obstacles

In [20]:
# Small-radius (0.35m) helix spiraling from z=0.25 to z=0.95 over 2.5 turns,
# staying close to the base -- every one of the 10 obstacles is >=0.58m from
# the base, so this curve clears all of them with margin (verified
# numerically: minimum clearance along the full continuous curve is ~4cm).
# Orientation points the tool mostly downward, tilted outward along the
# spiral -- a much more naturally reachable pose near the base than pointing
# straight out horizontally (which forces self-collision at this radius).
n_helix = 8
theta = np.linspace(0, 5 * np.pi, n_helix)
helix_radius = 0.35
helix_z = np.linspace(0.25, 0.95, n_helix)
world_up = np.array([0.0, 0.0, 1.0])

def helix_position(th, dth, z, dz):
    th2 = th + dth
    return np.array([helix_radius * np.cos(th2), helix_radius * np.sin(th2), z + dz])

def helix_orientation(th, dth, outward_weight, down_weight):
    th2 = th + dth
    outward = np.array([np.cos(th2), np.sin(th2), 0.0])
    approach = outward_weight * outward - down_weight * world_up
    n = np.linalg.norm(approach)
    if n < 1e-9:
        return None
    approach /= n
    right = np.cross(world_up, approach)
    rn = np.linalg.norm(right)
    if rn < 1e-6:
        return None
    right /= rn
    true_up = np.cross(approach, right)
    return np.stack([right, true_up, approach], axis=1)

# fallback search per point: try the exact curve position/orientation first,
# then a handful of downward/outward blends, then small along-curve nudges --
# keeps every waypoint as close as possible to the ideal smooth helix while
# guaranteeing it actually embeds (reachable + internal+external collision-free)
excluded_pairs = default_excluded_pairs(len(capsules))
orient_options = [(0.95, 0.3), (0.99, 0.15), (0.9, 0.45), (0.85, 0.55), (1.0, 0.0), (0.7, 0.7), (0.5, 0.85)]
jitter_options = [(0, 0), (0.15, 0), (-0.15, 0), (0.3, 0), (-0.3, 0), (0, 0.1), (0, -0.1)]

X_waypoints = []
for th, z in zip(theta, helix_z):
    found = False
    for dth, dz in jitter_options:
        p = helix_position(th, dth, z, dz)
        for down_weight, outward_weight in orient_options:
            R = helix_orientation(th, dth, outward_weight, down_weight)
            if R is None:
                continue
            _, _, node_idxs = embed_X_in_graph(
                [(p, R)], prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs,
                spheres=spheres, m=16,
            )
            if node_idxs[0]:
                X_waypoints.append((p, R))
                found = True
                break
        if found:
            break
    if not found:
        raise RuntimeError(f"no embeddable pose found near helix point theta={th:.2f}, z={z:.2f}")

print(f"{len(X_waypoints)}/{n_helix} helix waypoints ready")

8/8 helix waypoints ready


### Draw green arrows at the helix waypoints

In [21]:
for i, (p, R) in enumerate(X_waypoints):
    draw_pose_arrow(viz, f"waypoints/{i}", p, R)

In [23]:
# embed all helix waypoints together, then chain via Dijkstra -- each segment
# continues from EXACTLY the node the previous one arrived at (see the earlier
# chained-waypoint cell for why that matters)
aug_graph, aug_q_pool, node_idxs_per_X = embed_X_in_graph(
    X_waypoints, prm_graph_obstacles, q_pool_external_collision_free, model, data, capsules, excluded_pairs,
    spheres=spheres, m=16,
)
print("embedded nodes per waypoint:", [len(idxs) for idxs in node_idxs_per_X])

full_path = None
current_node = None
for a in node_idxs_per_X[0]:
    for b in node_idxs_per_X[1]:
        path, dist = dijkstra_path(aug_graph, a, b)
        if path is not None:
            full_path = list(path)
            current_node = b
            print(f"segment 0->1: {len(path)} nodes, distance {dist:.4f}")
            break
    if full_path is not None:
        break
if full_path is None:
    raise RuntimeError("segment 0->1 disconnected across every node candidate tried")

for i in range(1, len(X_waypoints) - 1):
    connected = False
    for b in node_idxs_per_X[i + 1]:
        path, dist = dijkstra_path(aug_graph, current_node, b)
        if path is not None:
            full_path.extend(path[1:])
            current_node = b
            print(f"segment {i}->{i + 1}: {len(path)} nodes, distance {dist:.4f}")
            connected = True
            break
    if not connected:
        raise RuntimeError(f"segment {i}->{i + 1} disconnected from arrival node {current_node} across every candidate tried")

print(f"total chained path: {len(full_path)} nodes")

use_shortcutting = False  # shortcutting can make the cubic spline overshoot through self-collision on sparse waypoints
q_path = aug_q_pool[full_path]
result = simulate_lqr_from_path(
    model, q_path, T=len(full_path) * 0.5, dt=0.01, viz=viz, viz_every=5, real_time=True,
    use_shortcutting=use_shortcutting, capsules=capsules, excluded_pairs=excluded_pairs, spheres=spheres,
)
print(f"simulated {len(result['t'])} steps, max tracking error {result['errors'].max():.4f}, final error {result['errors'][-1]:.4f}")

embedded nodes per waypoint: [2, 7, 9, 11, 3, 11, 10, 9]
segment 0->1: 27 nodes, distance 12.9365
segment 1->2: 11 nodes, distance 4.5966
segment 2->3: 17 nodes, distance 7.5382
segment 3->4: 18 nodes, distance 7.6130
segment 4->5: 19 nodes, distance 7.8826
segment 5->6: 10 nodes, distance 4.0632
segment 6->7: 21 nodes, distance 8.5172
total chained path: 117 nodes
simulated 5851 steps, max tracking error 0.0410, final error 0.0410
